In [0]:
import urllib.request
import xml.etree.ElementTree as ET
import pandas as pd

# Target coordinates: Live global energy market keyword search via open Google News RSS
url = "https://news.google.com/rss/search?q=global+energy+oil+gas+market&hl=en-US&gl=US&ceid=US:en"

try:
    # Ride out and capture the true live stream
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    response = urllib.request.urlopen(req)
    raw_payload = response.read()
    
    # Decipher the live XML payload
    xml_tree = ET.fromstring(raw_payload)
    
    live_bounty = []
    for item in xml_tree.findall('.//item'):
        title = item.find('title').text if item.find('title') is not None else ''
        link = item.find('link').text if item.find('link') is not None else ''
        pub_date = item.find('pubDate').text if item.find('pubDate') is not None else ''
        description = item.find('description').text if item.find('description') is not None else ''
        
        live_bounty.append({
            "title": title, 
            "link": link, 
            "published": pub_date, 
            "raw_text": description
        })
        
    # Build the true live Bronze DataFrame
    df = pd.DataFrame(live_bounty)
    print(f"[SUCCESS] Corralled {len(df)} completely REAL energy dispatches!")
    display(df.head())
    
except Exception as e:
    print(f"[AMBUSH] Failed to extract the live payload: {e}")

In [0]:
# --- SILVER LAYER PROCESSING ---
# Cleaning, structuring, and standardizing the raw text

# 1. Ensure no exact duplicate titles exist
df_silver = df.drop_duplicates(subset=['title']).copy()

# 2. Standardize timestamps to standard ISO format for data consistency
df_silver['published_parsed'] = pd.to_datetime(df_silver['published'], errors='coerce')

# 3. Clean up the raw text (strip whitespace, ensure string type)
df_silver['clean_text'] = df_silver['raw_text'].astype(str).str.strip()

# 4. Drop the old raw columns we no longer need to keep the schema tight
df_silver = df_silver[['title', 'link', 'published_parsed', 'clean_text']]

print(f"Silver Layer complete: Optimized {len(df_silver)} clean articles.")
display(df_silver.head())

In [0]:
# --- GOLD LAYER PROCESSING (LLM OPTIMIZATION) ---
# Transforming clean text into math vectors for LLM consumption

import hashlib
import numpy as np

def generate_mock_embedding(text, dimensions=8):
    """
    Simulates an embedding model by turning text into a repeatable 
    array of floating-point numbers representing semantic meaning.
    """
    # Create a stable hash from the text string
    hash_object = hashlib.sha256(text.encode('utf-8'))
    hash_hex = hash_object.hexdigest()
    
    # Generate numbers between -1.0 and 1.0 based on the hash pieces
    np.random.seed(int(hash_hex[:8], 16) % (2**32 - 1))
    vector = np.random.uniform(-1.0, 1.0, dimensions).round(4).tolist()
    return vector

# 1. Create a copy for our Gold presentation tier
df_gold = df_silver.copy()

# 2. Generate the Vector Embeddings column from our clean text
df_gold['vector_embedding'] = df_gold['clean_text'].apply(generate_mock_embedding)

# 3. Select final columns optimized for vector database ingestion
df_gold = df_gold[['title', 'clean_text', 'vector_embedding']]

print("Gold Layer complete: Vector database payload ready for LLM consumption.")
display(df_gold.head())

In [0]:
# --- WRITING TO THE GOLD DELTA LAKE TABLE ---
# Saving the engineered payload so a model can query it

try:
    # Convert the pandas DataFrame to a Spark DataFrame for native Databricks storage
    spark_df_gold = spark.createDataFrame(df_gold)
    
    # Save the data as a managed Delta table in your lakehouse
    # This physically writes the text and vectors to disk
    spark_df_gold.write.format("delta").mode("overwrite").saveAsTable("gold_energy_embeddings")
    
    print("[SUCCESS] Gold Delta table created! A model can now query this dataset.")
    
    # Verify by reading it back from the system
    display(spark.table("gold_energy_embeddings"))
    
except Exception as e:
    print(f"[ERROR] Could not write to table: {e}")

In [0]:
# File-Target: ~/.databricks_save_gold_table.py

In [0]:
# Check if the dataframe exists in active memory before writing
if 'spark_df_gold' in locals() or 'spark_df_gold' in globals():
    CATALOG_NAME = "energydata"
    SCHEMA_NAME = "default"
    TABLE_NAME = "gold_energy_news"

    full_table_path = f"{CATALOG_NAME}.{SCHEMA_NAME}.{TABLE_NAME}"

    # Save the dataframe as a delta table in Unity Catalog
    (spark_df_gold.write
     .mode("overwrite")
     .option("mergeSchema", "true")
     .saveAsTable(full_table_path))

    print(f"[SUCCESS] Table created at: {full_table_path}")
else:
    print("[ERROR] 'spark_df_gold' is missing from memory.")
    print("-> Please go to the top of your notebook and click 'Run All' or run the cells that clean the data first.")

In [0]:
from pyspark.sql.functions import monotonically_increasing_id

# Check if the dataframe exists in active memory before writing
if 'spark_df_gold' in locals() or 'spark_df_gold' in globals():
    CATALOG_NAME = "energydata"
    SCHEMA_NAME = "default"
    TABLE_NAME = "gold_energy_news"

    full_table_path = f"{CATALOG_NAME}.{SCHEMA_NAME}.{TABLE_NAME}"

    # Add a strict unique ID column to serve as the primary key
    final_df = spark_df_gold.withColumn("article_id", monotonically_increasing_id().cast("string"))

    # Save the dataframe as a delta table in Unity Catalog with the new column
    (final_df.write
     .mode("overwrite")
     .option("mergeSchema", "true")
     .saveAsTable(full_table_path))

    print(f"[SUCCESS] Table updated with primary key 'article_id' at: {full_table_path}")
else:
    print("[ERROR] 'spark_df_gold' is missing from memory.")
    print("-> Please run your upstream cells first to rebuild the dataframe.")

In [0]:
%sql
ALTER TABLE energydata.default.gold_energy_news 
SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

In [0]:
%sql
WITH matched_context AS (
  SELECT clean_text 
  FROM VECTOR_SEARCH(
    index => "energydata.default.gold_energy_news_index", 
    query_text => "What are the latest critical shifts and market pressures in the energy sector?", 
    num_results => 3
  )
)
SELECT ai_query(
  'databricks-meta-llama-3-3-70b-instruct',
  CONCAT(
    'Analyze these raw energy entries. Synthesize a concise executive summary based only on this text: ', 
    ARRAY_JOIN(COLLECT_LIST(clean_text), ' ')
  )
) AS energy_intelligence_report
FROM matched_context;

In [0]:
# Extract the full text from the automatically stored SQL dataframe variable
full_report = _sqldf.collect()[0]["energy_intelligence_report"]

# Clean display format
print(full_report)